In [1]:
"""
kaggle_train.py  —  SecureStego-UPI GAN Training
=================================================
Single self-contained script for Kaggle (GPU P100).

Kaggle setup:
  1. New notebook → Settings → Accelerator → GPU P100
  2. Add Data → search "celeba" → add "CelebFaces Attributes (CelebA) Dataset"
     (by Jessica Li)
  3. Paste this entire script into one code cell and run.

Output:
  /kaggle/working/checkpoint_final.pt   ← download this when done
  /kaggle/working/samples/              ← face images generated every 5 epochs
  /kaggle/working/history.json          ← loss + BER per epoch

WHY ANNOTATIONS ARE NOT USED
──────────────────────────────
CelebA ships with attribute CSVs (smiling, glasses, etc.) and landmark
coordinates. None of those are needed here because our GAN is fully
unsupervised: the Discriminator only asks "real face or generated face?"
and never looks at any label.

The ONE annotation file we DO use is list_eval_partition.csv.  It marks
every image as train (0), val (1), or test (2).  We train only on the
official 162,770 training images rather than all 202,599, which is the
academically correct split and avoids leaking test images into training.
If the file is absent we fall back to using all images.
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────

import os, json, time, csv
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.utils import save_image
from PIL import Image

# ── 1. Config ─────────────────────────────────────────────────────────────────

class Config:
    # Image
    image_size     = 64
    image_channels = 3

    # Payload: IV(16 bytes) + Ciphertext(16 bytes) = 32 bytes = 256 bits
    payload_bits   = 256

    # Architecture
    noise_dim      = 100
    gen_features   = 64
    dis_features   = 64
    ext_features   = 64

    # ── Two-phase training ────────────────────────────────────────────────────
    #
    # PHASE 1 — Steganography warm-up (no Discriminator)
    #   G and E train together on reconstruction loss ONLY.
    #   G learns to embed bits. E learns to extract them.
    #   No image realism pressure yet — BER must fall before we add D.
    #   Target: BER < 0.10 by the end of phase 1.
    #
    # PHASE 2 — Joint GAN training (G + D + E together)
    #   D is introduced to push G toward realistic images.
    #   D is updated every d_update_freq batches to prevent it dominating G.
    #   Reconstruction loss keeps a high weight so BER keeps improving.
    #
    warmup_epochs   = 10     # phase 1: G+E only, no D
    gan_epochs      = 40     # phase 2: full G+D+E
    d_update_freq   = 3      # update D every N batches (slows D down)

    # Learning rates
    lr_G            = 2e-4
    lr_D            = 1e-4   # D is slower than G on purpose
    lr_E            = 1e-4
    beta1           = 0.5
    beta2           = 0.999

    # Loss weights
    # alpha: adversarial loss weight (image realism) — only active in phase 2
    # beta_recon: reconstruction loss weight (message recovery)
    # Keep beta_recon high so BER continues to improve during phase 2
    alpha           = 0.1    # low — realism is secondary to message recovery
    beta_recon      = 50.0   # high — message recovery is the primary objective

    batch_size      = 128
    use_amp         = True

    # Paths
    celeba_root     = "/kaggle/input/datasets/jessicali9530/celeba-dataset/"
    output_dir      = "/kaggle/working"
    sample_dir      = "/kaggle/working/samples"

    @property
    def num_epochs(self):
        return self.warmup_epochs + self.gan_epochs

    @property
    def device(self):
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg = Config()

# ── 2. Dataset ────────────────────────────────────────────────────────────────

def find_image_dir(root: str) -> Path:
    """
    Locate the folder that actually contains the .jpg face images.

    The Kaggle CelebA dataset (jessicali9530/celeba-dataset) has this layout:
        /kaggle/input/celeba-dataset/
            img_align_celeba/
                img_align_celeba/       ← images are usually one level deeper
                    000001.jpg
                    ...
            list_attr_celeba.csv
            list_eval_partition.csv
            ...

    We probe both possibilities and return whichever contains .jpg files.
    """
    root = Path(root)
    candidates = [
        root / "img_align_celeba" / "img_align_celeba",   # most common on Kaggle
        root / "img_align_celeba",                          # sometimes one level
    ]
    for c in candidates:
        if c.is_dir() and any(c.glob("*.jpg")):
            return c
    raise FileNotFoundError(
        f"Could not find face images under {root}.\n"
        "Expected .jpg files at:\n"
        f"  {candidates[0]}\n  or\n  {candidates[1]}\n"
        "Make sure you added the CelebA dataset in Kaggle."
    )


def load_train_filenames(root: str, image_dir: Path) -> list | None:
    """
    Read list_eval_partition.csv and return only the filenames marked as
    partition 0 (official training split = 162,770 images).

    Returns None if the CSV is not found, which causes the dataset to use
    all images instead.
    """
    csv_path = Path(root) / "list_eval_partition.csv"
    if not csv_path.exists():
        print("  list_eval_partition.csv not found — using all images.")
        return None

    train_files = []
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if int(row["partition"]) == 0:
                train_files.append(row["image_id"])

    # Verify filenames match actual images
    found = sum(1 for fn in train_files[:10] if (image_dir / fn).exists())
    if found == 0:
        print("  Train filenames from CSV don't match image dir — using all images.")
        return None

    return train_files


class CelebATrainDataset(Dataset):
    """
    Loads CelebA face images for GAN training.

    If train_filenames is provided: loads only the official 162k train split.
    Otherwise: loads every .jpg in image_dir.

    Returns (image_tensor, 0). The label 0 is a dummy — the GAN never uses it.
    """
    def __init__(self, image_dir: Path, transform, train_filenames=None):
        if train_filenames is not None:
            self.paths = [image_dir / fn for fn in train_filenames
                          if (image_dir / fn).exists()]
        else:
            self.paths = sorted(
                p for p in image_dir.iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
        if not self.paths:
            raise RuntimeError(f"No images found in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert("RGB")), 0


def make_transform():
    return transforms.Compose([
        transforms.Resize(cfg.image_size, interpolation=InterpolationMode.BILINEAR),
        transforms.CenterCrop(cfg.image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),   # → [-1, 1]
    ])


def build_dataloader() -> DataLoader:
    print("Locating CelebA images...")
    image_dir = find_image_dir(cfg.celeba_root)
    print(f"  Image dir : {image_dir}")

    print("Loading train split from list_eval_partition.csv...")
    train_files = load_train_filenames(cfg.celeba_root, image_dir)

    dataset = CelebATrainDataset(image_dir, make_transform(), train_files)

    split_label = "official train split" if train_files else "all images"
    print(f"  Images    : {len(dataset):,}  ({split_label})\n")

    return DataLoader(
        dataset,
        batch_size      = cfg.batch_size,
        shuffle         = True,
        num_workers     = 4,
        pin_memory      = True,
        drop_last       = True,
        prefetch_factor = 2,
    )


# ── 3. Models ─────────────────────────────────────────────────────────────────

def _init_weights(m):
    name = type(m).__name__
    if "Conv" in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    """
    Encoder-Generator G.
    Input : [z (noise) || m (message bits)]  →  B × (noise_dim + payload_bits)
    Output: synthetic face image              →  B × 3 × 64 × 64, values in [-1, 1]

    Upsampling: 1×1 → 4×4 → 8×8 → 16×16 → 32×32 → 64×64
    """
    def __init__(self):
        super().__init__()
        latent = cfg.noise_dim + cfg.payload_bits
        nf     = cfg.gen_features
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent, nf*8, 4, 1, 0, bias=False), nn.BatchNorm2d(nf*8), nn.ReLU(True),
            nn.ConvTranspose2d(nf*8,  nf*4, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*4), nn.ReLU(True),
            nn.ConvTranspose2d(nf*4,  nf*2, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*2), nn.ReLU(True),
            nn.ConvTranspose2d(nf*2,  nf,   4, 2, 1, bias=False), nn.BatchNorm2d(nf),   nn.ReLU(True),
            nn.ConvTranspose2d(nf, cfg.image_channels, 4, 2, 1,   bias=False),           nn.Tanh(),
        )
        self.apply(_init_weights)

    def forward(self, z, m):
        return self.net(torch.cat([z, m], 1).unsqueeze(-1).unsqueeze(-1))


class Discriminator(nn.Module):
    """
    Discriminator D.
    Input : image  B × 3 × 64 × 64
    Output: raw logits  B  (NOT passed through Sigmoid)

    We removed Sigmoid deliberately. BCEWithLogitsLoss fuses sigmoid+BCE
    internally in a numerically stable way that is safe under AMP/float16.
    Using nn.Sigmoid() here followed by BCELoss crashes under autocast
    because the sigmoid output in float16 can saturate to exactly 0 or 1,
    making log(0) = -inf in the BCE calculation.
    Only used during training; discarded at inference.
    """
    def __init__(self):
        super().__init__()
        nf = cfg.dis_features
        self.net = nn.Sequential(
            nn.Conv2d(cfg.image_channels, nf,   4, 2, 1, bias=False),                     nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf,   nf*2, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*2),             nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*2, nf*4, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*4),             nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*4, nf*8, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*8),             nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*8, 1,    4, 1, 0, bias=False),
            # NO Sigmoid here — BCEWithLogitsLoss handles it safely
        )
        self.apply(_init_weights)

    def forward(self, x):
        return self.net(x).view(-1)


class Extractor(nn.Module):
    """
    Extractor E.
    Input : stego image  B × 3 × 64 × 64
    Output: message logits  B × payload_bits
            Apply sigmoid + threshold >= 0.5 to recover binary bits.
    """
    def __init__(self):
        super().__init__()
        nf = cfg.ext_features
        self.conv = nn.Sequential(
            nn.Conv2d(cfg.image_channels, nf,   4, 2, 1, bias=False),                     nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf,   nf*2, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*2),             nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*2, nf*4, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*4),             nn.LeakyReLU(0.2, True),
            nn.Conv2d(nf*4, nf*8, 4, 2, 1, bias=False), nn.BatchNorm2d(nf*8),             nn.LeakyReLU(0.2, True),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(nf*8*4*4, 512), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(512, cfg.payload_bits),
        )
        self.apply(_init_weights)

    def forward(self, x):
        return self.fc(self.conv(x))


# ── 4. Training utilities ─────────────────────────────────────────────────────

def rand_bits(b):
    return torch.randint(0, 2, (b, cfg.payload_bits),
                         dtype=torch.float32, device=cfg.device)

def rand_z(b):
    return torch.randn(b, cfg.noise_dim, device=cfg.device)


@torch.no_grad()
def compute_ber(G, E, n_batches: int = 8) -> float:
    """Bit Error Rate: fraction of bits incorrectly recovered. Target < 0.05."""
    G.eval(); E.eval()
    total = wrong = 0
    for _ in range(n_batches):
        b     = cfg.batch_size
        z, m  = rand_z(b), rand_bits(b)
        m_hat = (torch.sigmoid(E(G(z, m))) >= 0.5).float()
        wrong += (m_hat != m).sum().item()
        total += m.numel()
    G.train(); E.train()
    return wrong / total


# ── 5. Main training loop ─────────────────────────────────────────────────────

def train():
    device = cfg.device
    print("=" * 62)
    print("  SecureStego-UPI — GAN Training (Kaggle)")
    print("=" * 62)
    print(f"  Device         : {device}")
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        print(f"  GPU            : {props.name}")
        print(f"  VRAM           : {props.total_memory / 1e9:.1f} GB")
    print(f"  AMP            : {cfg.use_amp}")
    print(f"  Batch size     : {cfg.batch_size}")
    print(f"  Phase 1 epochs : {cfg.warmup_epochs}  (G+E only, no D)")
    print(f"  Phase 2 epochs : {cfg.gan_epochs}  (G+D+E joint)")
    print(f"  D update every : {cfg.d_update_freq} batches")
    print(f"  alpha / beta   : {cfg.alpha} / {cfg.beta_recon}")
    print("=" * 62 + "\n")

    Path(cfg.sample_dir).mkdir(parents=True, exist_ok=True)

    loader    = build_dataloader()
    n_batches = len(loader)
    total_ep  = cfg.num_epochs
    print(f"Batches per epoch : {n_batches}")
    print(f"Total epochs      : {total_ep}  ({cfg.warmup_epochs} warmup + {cfg.gan_epochs} GAN)\n")

    G = Generator().to(device)
    D = Discriminator().to(device)
    E = Extractor().to(device)

    total_params = sum(p.numel() for model in [G, D, E] for p in model.parameters())
    print(f"Total parameters  : {total_params:,}\n")

    opt_G = optim.Adam(G.parameters(), lr=cfg.lr_G, betas=(cfg.beta1, cfg.beta2))
    opt_D = optim.Adam(D.parameters(), lr=cfg.lr_D, betas=(cfg.beta1, cfg.beta2))
    opt_E = optim.Adam(E.parameters(), lr=cfg.lr_E, betas=(cfg.beta1, cfg.beta2))

    bce_logits = nn.BCEWithLogitsLoss()
    scaler     = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    history  = []
    total_t0 = time.time()

    for epoch in range(1, total_ep + 1):
        epoch_t0 = time.time()
        phase    = 1 if epoch <= cfg.warmup_epochs else 2
        sum_D = sum_G = sum_R = 0.0
        d_updates = 0

        if epoch == 1:
            print("── Phase 1: Steganography warm-up (G + E only) ─────────────────")
            print("   D is frozen. G learns to embed bits. E learns to extract them.")
            print("   Watch Recon fall from 0.693. BER must drop before phase 2.\n")
        elif epoch == cfg.warmup_epochs + 1:
            print("\n── Phase 2: Joint GAN training (G + D + E) ─────────────────────")
            print("   D added to push G toward realistic images.")
            print(f"  D updates every {cfg.d_update_freq} batches to avoid dominating G.\n")

        for i, (real_imgs, _) in enumerate(loader):
            real_imgs = real_imgs.to(device, non_blocking=True)
            b = real_imgs.size(0)

            if phase == 2 and (i % cfg.d_update_freq == 0):
                # ── Train D (phase 2 only, throttled) ────────────────────────
                with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                    with torch.no_grad():
                        fake_d = G(rand_z(b), rand_bits(b))
                    loss_D = (
                        bce_logits(D(real_imgs), torch.ones(b,  device=device)) +
                        bce_logits(D(fake_d),    torch.zeros(b, device=device))
                    ) * 0.5
                opt_D.zero_grad()
                scaler.scale(loss_D).backward()
                scaler.step(opt_D)
                sum_D += loss_D.item()
                d_updates += 1

            # ── Train G + E (both phases) ─────────────────────────────────────
            z, m = rand_z(b), rand_bits(b)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                fake       = G(z, m)
                loss_recon = bce_logits(E(fake), m)

                if phase == 2:
                    # Adversarial loss: G wants D to classify fake as real
                    loss_adv = bce_logits(D(fake), torch.ones(b, device=device))
                    loss_GE  = cfg.alpha * loss_adv + cfg.beta_recon * loss_recon
                else:
                    # Phase 1: pure reconstruction — no adversarial signal
                    loss_adv = torch.tensor(0.0, device=device)
                    loss_GE  = cfg.beta_recon * loss_recon

            opt_G.zero_grad(); opt_E.zero_grad()
            scaler.scale(loss_GE).backward()
            scaler.step(opt_G); scaler.step(opt_E)
            scaler.update()

            sum_G += loss_adv.item()
            sum_R += loss_recon.item()

            if (i + 1) % 200 == 0:
                done    = (i + 1) / n_batches
                eta_min = (time.time() - epoch_t0) * (1 - done) / done / 60
                d_str   = f"D={sum_D/max(d_updates,1):.4f}  " if phase == 2 else ""
                print(f"  [P{phase} {epoch}/{total_ep}] "
                      f"batch {i+1}/{n_batches}  "
                      f"{d_str}"
                      f"G={sum_G/(i+1):.4f}  "
                      f"Recon={sum_R/(i+1):.4f}  "
                      f"ETA: {eta_min:.1f}min")

        # ── Epoch summary ─────────────────────────────────────────────────────
        ber      = compute_ber(G, E)
        ep_secs  = time.time() - epoch_t0
        total_eta = (time.time() - total_t0) / epoch * (total_ep - epoch)

        rec = {
            "epoch":      epoch,
            "phase":      phase,
            "loss_D":     round(sum_D / max(d_updates, 1), 4),
            "loss_G":     round(sum_G / n_batches, 4),
            "loss_recon": round(sum_R / n_batches, 4),
            "ber":        round(ber, 4),
            "secs":       round(ep_secs, 1),
        }
        history.append(rec)

        ber_tag = "TARGET MET" if ber < 0.05 else ("good" if ber < 0.10 else "training...")
        print(f"\n[P{phase}] Epoch {epoch:2d}/{total_ep}  "
              f"Recon={rec['loss_recon']}  BER={rec['ber']} ({ber_tag})  "
              f"({ep_secs/60:.1f}min  ETA {total_eta/3600:.1f}h)\n")

        if epoch % 5 == 0:
            G.eval()
            with torch.no_grad():
                samples = (G(rand_z(16), rand_bits(16)).clamp(-1, 1) + 1) / 2
                save_image(samples, f"{cfg.sample_dir}/epoch_{epoch:03d}.png", nrow=4)
            G.train()
            ck = f"{cfg.output_dir}/checkpoint_{epoch}.pt"
            torch.save({"G": G.state_dict(), "D": D.state_dict(),
                        "E": E.state_dict(), "epoch": epoch,
                        "history": history}, ck)
            print(f"  Checkpoint: {ck}")

        with open(f"{cfg.output_dir}/history.json", "w") as f:
            json.dump(history, f, indent=2)

    # ── Final save ────────────────────────────────────────────────────────────
    final = f"{cfg.output_dir}/checkpoint_final.pt"
    torch.save({"G": G.state_dict(), "D": D.state_dict(),
                "E": E.state_dict(), "epoch": total_ep,
                "history": history}, final)

    total_min = (time.time() - total_t0) / 60
    print(f"\n{'='*62}")
    print(f"  Training complete in {total_min:.1f} minutes.")
    print(f"  Final BER  : {history[-1]['ber']:.4f}  (target < 0.05)")
    print(f"  Checkpoint : {final}")
    print(f"  Download checkpoint_final.pt from the Output panel on the right.")
    print(f"{'='*62}\n")

    # ── Sanity check ──────────────────────────────────────────────────────────
    print("Post-training sanity check...")
    G.eval(); E.eval()
    with torch.no_grad():
        z, m      = rand_z(16), rand_bits(16)
        imgs      = G(z, m)
        bits      = (torch.sigmoid(E(imgs)) >= 0.5).float()
        final_ber = (bits != m).float().mean().item()

    print(f"  BER on 16 samples : {final_ber:.4f}")
    if final_ber < 0.05:
        print("  Model is ready to use.")
    elif final_ber < 0.10:
        print("  Acceptable. Run 10 more phase-2 epochs if you want sharper recovery.")
    else:
        print("  BER still above 0.10.")
        print("  If Phase 1 Recon never dropped below 0.4: increase warmup_epochs to 15.")
        print("  If Phase 1 worked but Phase 2 broke it: lower alpha to 0.05.")



# ── 6. Entry point ────────────────────────────────────────────────────────────

if __name__ == "__main__":
    train()

  SecureStego-UPI — GAN Training (Kaggle)
  Device         : cuda
  GPU            : Tesla T4
  VRAM           : 15.6 GB
  AMP            : True
  Batch size     : 128
  Phase 1 epochs : 10  (G+E only, no D)
  Phase 2 epochs : 40  (G+D+E joint)
  D update every : 3 batches
  alpha / beta   : 0.1 / 50.0

Locating CelebA images...
  Image dir : /kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba
Loading train split from list_eval_partition.csv...
  Images    : 162,770  (official train split)

Batches per epoch : 1271
Total epochs      : 50  (10 warmup + 40 GAN)

Total parameters  : 15,522,944

── Phase 1: Steganography warm-up (G + E only) ─────────────────
   D is frozen. G learns to embed bits. E learns to extract them.
   Watch Recon fall from 0.693. BER must drop before phase 2.

  [P1 1/50] batch 200/1271  G=0.0000  Recon=0.6178  ETA: 4.3min
  [P1 1/50] batch 400/1271  G=0.0000  Recon=0.5418  ETA: 3.5min
  [P1 1/50] batch 600/1271  G=0.0000  Recon=0